# LoRA vs full fine-tuning

Same model, same data, same number of epochs. Only the training method changes. I use DistilBERT on a 2,000-example slice of SST-2 so each run takes a few minutes, even on a T4.

The learning rates are different on purpose. LoRA usually wants a much higher one than full fine-tuning.

In [ ]:
import time

import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

MODEL = "distilbert-base-uncased"
N_TRAIN = 2000
EPOCHS = 3
cuda = torch.cuda.is_available()
print("cuda:", cuda)

In [ ]:
raw = load_dataset("glue", "sst2")
tok = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tok(batch["sentence"], truncation=True, max_length=128)

train = raw["train"].shuffle(seed=42).select(range(N_TRAIN))
train = train.map(tokenize, batched=True, remove_columns=["sentence", "idx"])
val = raw["validation"].map(tokenize, batched=True, remove_columns=["sentence", "idx"])
print(len(train), "train /", len(val), "validation")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": float((logits.argmax(-1) == labels).mean())}


def run(method):
    set_seed(42)
    if cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
    if method == "lora":
        config = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16,
                            lora_dropout=0.05, target_modules=["q_lin", "v_lin"])
        model = get_peft_model(model, config)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())

    args = TrainingArguments(
        output_dir=f"outputs/{method}",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        learning_rate=2e-4 if method == "lora" else 2e-5,
        warmup_ratio=0.1,
        eval_strategy="no",
        save_strategy="no",
        fp16=cuda,
        logging_steps=20,
        report_to="none",
        seed=42,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=val,
                      tokenizer=tok, data_collator=DataCollatorWithPadding(tok),
                      compute_metrics=compute_metrics)

    start = time.time()
    trainer.train()
    seconds = time.time() - start
    acc = trainer.evaluate()["eval_accuracy"]

    return {
        "method": method,
        "trainable_params": trainable,
        "pct_trainable": round(100 * trainable / total, 2),
        "train_seconds": round(seconds),
        "peak_gpu_mb": round(torch.cuda.max_memory_allocated() / 1e6) if cuda else None,
        "val_accuracy": round(acc, 4),
    }

In [ ]:
results = [run("full"), run("lora")]
df = pd.DataFrame(results)
df

In [ ]:
import os
os.makedirs("results", exist_ok=True)
df.to_csv("results/lora_vs_full.csv", index=False)

## Things to try next

- Sweep the LoRA rank (r = 1, 4, 8, 16, 64) and see where accuracy stops improving.
- Put LoRA on more layers (`k_lin`, `out_lin`, the FFN layers).
- Train on less data. LoRA often holds up better when there are only a few hundred examples.
- Run each method with a few seeds. One run on 2,000 examples is noisy, and small gaps could just be luck.